In [ ]:
"""
# EAHN — Kaggle Training Notebook (Two-Phase: Synthetic Pre-train + FF++ Fine-tune)
# 
# INSTRUCTIONS:
# 1. Create new Kaggle notebook with GPU (T4) accelerator
# 2. Enable Internet in Settings (right panel)
# 3. Add your FF++ c23 dataset to Input
# 4. Run all cells top-to-bottom
# 5. After Phase 2 Epoch 2, check train_metrics.csv. If Val AUC > 0.65, change EPOCHS_PHASE2 to 15 and re-run from Cell 7.
"""

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 1 — Configuration & Paths                                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║ CELL 1 — CONFIGURATION                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, glob

def find_ffpp_root(base="/kaggle/input", max_depth=6):
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        if (os.path.isdir(os.path.join(root, "original_sequences")) and
            os.path.isdir(os.path.join(root, "manipulated_sequences"))):
            return root
    return None

DATA_ROOT = find_ffpp_root()

REPO_URL     = "https://github.com/umardrazbhatti/EAHNKimiCode.git"
REPO_DIR     = "/kaggle/working/EAHNKimiCode"
OUTPUT_DIR   = "/kaggle/working/outputs"
CACHE_DIR    = "/kaggle/working/.face_cache"
SYNTH_SOURCE = "/kaggle/working/synth_source"

EPOCHS_PHASE1 = 10   # Synthetic pre-training
EPOCHS_PHASE2 = 2    # FF++ smoke test (bump to 15 after pass)

# FIX: Reduced for Kaggle T4 (16 GB). Scripts auto-adjust if you forget,
# but explicit is safer. Effective batch = 2 * 8 = 16.
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8

if DATA_ROOT is None:
    print("ERROR: FF++ root not found. Listing /kaggle/input for diagnosis:")
    for e in sorted(glob.glob("/kaggle/input/*/*")):
        print(f"  {e}")
    raise SystemExit("Set DATA_ROOT manually above after checking the listing.")

print("=" * 60)
print(f"DATA_ROOT      : {DATA_ROOT}")
print(f"REPO_DIR       : {REPO_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
print(f"SYNTH_SOURCE   : {SYNTH_SOURCE}")
print(f"EPOCHS_PHASE1  : {EPOCHS_PHASE1} (synthetic pre-training)")
print(f"EPOCHS_PHASE2  : {EPOCHS_PHASE2} (FF++ fine-tuning)")
print(f"BATCH_SIZE     : {BATCH_SIZE}")
print(f"GRAD_ACCUM_STEPS: {GRAD_ACCUM_STEPS} (effective batch = {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print("=" * 60)
print("If DATA_ROOT looks wrong, set it manually and re-run only this cell.")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 2 — Verify Dataset Structure                                            ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import glob, os

In [ ]:
METHODS = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]

In [ ]:
real_dir = os.path.join(DATA_ROOT, "original_sequences", "youtube", "c23", "videos")
real_vids = glob.glob(os.path.join(real_dir, "*.mp4"))
status = "OK" if len(real_vids) > 0 else "MISSING"
print(f"[{status:^7}] {'original (real)':<22} {len(real_vids):>5} videos")

In [ ]:
total_fake, all_ok = 0, len(real_vids) > 0
for method in METHODS:
    d = os.path.join(DATA_ROOT, "manipulated_sequences", method, "c23", "videos")
    count = len(glob.glob(os.path.join(d, "*.mp4")))
    total_fake += count
    status = "OK" if count > 0 else "MISSING"
    if count == 0: all_ok = False
    print(f"[{status:^7}] {method:<22} {count:>5} videos")

In [ ]:
print(f"\nTotal real: {len(real_vids)} | Total fake: {total_fake} | Combined: {len(real_vids) + total_fake}")
if not all_ok:
    raise SystemExit("One or more directories missing. Fix DATA_ROOT in Cell 1.")
print("\nFile counts OK --- proceed to Cell 3.")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 3 — Clean Working Directories                                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import shutil, os

In [ ]:
to_remove = [REPO_DIR, OUTPUT_DIR, CACHE_DIR, "/kaggle/working/eahn_outputs.zip"]
for path in to_remove:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Removed dir : {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Removed file: {path}")
    else:
        print(f"Not present : {path}")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SYNTH_SOURCE, exist_ok=True)
print("\nClean. Proceed to Cell 4.")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 4 — Clone Repository                                                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os, sys

In [ ]:
ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
if ret != 0 or not os.path.isdir(REPO_DIR):
    raise SystemExit(
        "Clone failed.\n"
        "Fix: Enable Internet in Kaggle Settings (right panel → Internet → On).\n"
        "Then re-run this cell."
    )

In [ ]:
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
print("\nClone successful. Repo contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(f"  {f}")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 5 — Install Dependencies & Verify Imports                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import subprocess, sys, importlib

In [ ]:
print("Installing requirements...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     f"{REPO_DIR}/requirements.txt"],
    check=True
)
print("Installation complete.\n")

In [ ]:
pkg_map = {"torch":"torch","torchvision":"torchvision","timm":"timm",
           "sklearn":"sklearn","cv2":"cv2","PIL":"PIL","tqdm":"tqdm",
           "gradcam":"pytorch_grad_cam", "captum":"captum"}
for name, mod_name in pkg_map.items():
    try:
        m = importlib.import_module(mod_name)
        print(f" OK {name:<15} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f" MISSING {name}")

In [ ]:
print("\nChecking full import chain...")
try:
    from config import EAHNConfig
    from data.datasets import DeepfakeDataset
    from data.collate import deepfake_collate_fn
    from data.transforms import get_transforms
    from data.face_align import FaceAligner
    from data.synthetic_generator import SyntheticDataGenerator, SyntheticDataset
    from models.eahn import EAHN
    from losses.classification import build_classification_loss
    from losses.explanation import ExplanationLoss
    from losses.temporal import TemporalConsistencyLoss
    from metrics.detection import DetectionMetrics
    from scripts.train_synthetic import main as train_synth_main
    from scripts.train_real import main as train_real_main
    from scripts.evaluate import run_evaluation
    print("\nALL IMPORTS OK --- proceed to Cell 6.")
except ImportError as e:
    raise SystemExit(f"\nIMPORT FAILED: {e}")

In [ ]:
# Verify config has new two-phase defaults
config = EAHNConfig()
assert hasattr(config, 'attn_temp_init'), "attn_temp_init missing from EAHNConfig"
assert hasattr(config, 'attn_diversity_weight'), "attn_diversity_weight missing from EAHNConfig"
assert config.gamma == 0.1, f"gamma must be 0.1, got {config.gamma}"
print(f"Config verified: gamma={config.gamma}, temp_init={config.attn_temp_init}, diversity_w={config.attn_diversity_weight}")
print(f"Config: alpha={config.alpha}, lambda2={config.lambda2}, cls_loss_type={config.cls_loss_type}")
print(f"Config: freeze_backbone={config.freeze_backbone}, lr={config.lr}, backbone_lr_ratio={config.backbone_lr_ratio}")
print(f"Config: lambda1={config.lambda1} (should be 0.05 for FF++ phase, 1.0 for synthetic phase)")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 6 — Extract Source Faces for Synthetic Generation                       ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import cv2
from tqdm import tqdm

In [ ]:
print("Extracting source faces from real FF++ videos for synthetic pre-training...")
real_videos = glob.glob(f"{DATA_ROOT}/original_sequences/youtube/c23/videos/*.mp4")
extracted = 0

In [ ]:
for vid in tqdm(real_videos, desc="Extracting faces"):
    cap = cv2.VideoCapture(vid)
    ret, frame = cap.read()
    if ret and frame is not None:
        h, w = frame.shape[:2]
        # Center crop face region (rough heuristic for FF++ aligned faces)
        face = frame[max(0, h//4):min(h, 3*h//4), max(0, w//4):min(w, 3*w//4)]
        if face.size > 0:
            face = cv2.resize(face, (224, 224))
            out_name = os.path.basename(vid).replace(".mp4", ".jpg")
            cv2.imwrite(os.path.join(SYNTH_SOURCE, out_name), face)
            extracted += 1
    cap.release()

In [ ]:
print(f"\nExtracted {extracted} source face crops to {SYNTH_SOURCE}")
if extracted < 50:
    raise SystemExit(f"Too few source images ({extracted}). Check DATA_ROOT and video paths.")
print("Source faces ready. Proceed to Phase 1 (Cell 7).")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 7 — PHASE 1: Synthetic Pre-Training (10 epochs)                        ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os, sys, shutil

In [ ]:
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
# Clear face cache to prevent any collision
if os.path.isdir(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print("Face cache cleared for synthetic phase.")

In [ ]:
cmd = (
    f'python scripts/train_synthetic.py'
    f' --output_dir "{OUTPUT_DIR}"'
    f' --cache_dir "{CACHE_DIR}"'
    f' --epochs {EPOCHS_PHASE1}'
    f' --batch_size {BATCH_SIZE}'
    f' --num_workers 0'
    f' --lr 5e-4'
    f' --lambda1 1.0'
    f' --lambda2 0.1'
    f' --alpha 0.5'
    f' --beta 0.5'
    f' --gamma 0.1'
    f' --attn_temp_init 0.693'
    f' --attn_diversity_weight 8.0'
    f' --cls_loss_type focal'
    f' --focal_alpha 0.25'
    f' --focal_gamma 2.0'
    f' --label_smoothing 0.02'
    f' --grad_accum_steps {GRAD_ACCUM_STEPS}'   # <-- was hardcoded 4, now uses variable
    f' --warmup_epochs 2'
    f' --patience 5'
    f' --eval_after_train'
)

In [ ]:
print("\n" + "="*60)
print("PHASE 1: Synthetic Pre-Training")
print("="*60)
print("Running:", cmd, "\n")
os.system(cmd)
print("\nCell 7 (Phase 1) complete.")

In [ ]:
# Verify checkpoint was saved
synth_ckpt = os.path.join(OUTPUT_DIR, "synthetic_pretrained.pth")
if not os.path.exists(synth_ckpt):
    raise SystemExit(f"\nCRITICAL: Synthetic checkpoint not found at {synth_ckpt}\n"
                     "Phase 1 failed. Check logs above for errors.")
print(f"\n✅ Synthetic checkpoint verified: {synth_ckpt}")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 8 — PHASE 2: FF++ Fine-Tuning (2 epochs smoke test)                     ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os, sys, shutil

In [ ]:
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
# Clear face cache for FF++ phase (different num_frames may be cached)
if os.path.isdir(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print("Face cache cleared for FF++ phase.")

In [ ]:
cmd = (
    f'python scripts/train_real.py'
    f' --data_root "{DATA_ROOT}"'
    f' --dataset_name ff++'
    f' --output_dir "{OUTPUT_DIR}"'
    f' --cache_dir "{CACHE_DIR}"'
    f' --epochs {EPOCHS_PHASE2}'
    f' --batch_size {BATCH_SIZE}'
    f' --num_workers 0'
    f' --lr 5e-4'
    f' --backbone_lr_ratio 1.0'
    f' --lambda1 0.05'
    f' --lambda2 0.02'
    f' --grad_accum_steps {GRAD_ACCUM_STEPS}'   # <-- same fix
    f' --eval_after_train'
)

In [ ]:
print("\n" + "="*60)
print("PHASE 2: FF++ Fine-Tuning (Smoke Test: 2 epochs)")
print("="*60)
print("Running:", cmd, "\n")
os.system(cmd)
print("\nCell 8 (Phase 2) complete.")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 9 — Smoke Test Analysis (Check before running 15 epochs)                ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import pandas as pd, os

In [ ]:
train_csv = os.path.join(OUTPUT_DIR, "train_metrics.csv")
metrics_csv = os.path.join(OUTPUT_DIR, "metrics.csv")

In [ ]:
print("\n" + "="*60)
print("SMOKE TEST ANALYSIS")
print("="*60)

In [ ]:
# Check training metrics
if os.path.exists(train_csv):
    df = pd.read_csv(train_csv)
    print("\n📊 Training Metrics (train_metrics.csv):")
    print(df.to_string(index=False))

    # Pass criteria checks
    checks_passed = 0
    checks_total = 4

    if len(df) >= 2:
        epoch2 = df.iloc[-1]

        # Check 1: Val AUC > 0.60
        val_auc = epoch2.get('val_auc', 0)
        if val_auc > 0.60:
            print(f"\n✅ PASS: Val AUC = {val_auc:.4f} (> 0.60)")
            checks_passed += 1
        else:
            print(f"\n❌ FAIL: Val AUC = {val_auc:.4f} (need > 0.60)")
            print("   → Model is not learning. Check: lr too low? lambda1 too high?")

        # Check 2: Train loss decreasing
        if len(df) >= 2:
            loss_trend = df['train_loss'].iloc[-1] < df['train_loss'].iloc[0]
            if loss_trend:
                print(f"✅ PASS: Train loss decreasing ({df['train_loss'].iloc[0]:.4f} → {df['train_loss'].iloc[-1]:.4f})")
                checks_passed += 1
            else:
                print(f"❌ FAIL: Train loss INCREASING ({df['train_loss'].iloc[0]:.4f} → {df['train_loss'].iloc[-1]:.4f})")
                print("   → Loss exploding. Reduce lr to 1e-4 or check for NaN.")

        # Check 3: Val Real Acc > 0.50
        val_real_acc = epoch2.get('val_real_acc', 0)
        if val_real_acc > 0.50:
            print(f"✅ PASS: Val Real Acc = {val_real_acc:.3f} (> 0.50)")
            checks_passed += 1
        else:
            print(f"❌ FAIL: Val Real Acc = {val_real_acc:.3f} (need > 0.50)")
            print("   → Model ignoring minority class. Check stratified sampler.")

        # Check 4: L_cls vs L_exp balance (indirect: check if train_fake_acc reasonable)
        train_fake_acc = epoch2.get('train_fake_acc', 0)
        if train_fake_acc > 0.50:
            print(f"✅ PASS: Train Fake Acc = {train_fake_acc:.3f} (> 0.50)")
            checks_passed += 1
        else:
            print(f"❌ FAIL: Train Fake Acc = {train_fake_acc:.3f} (need > 0.50)")
            print("   → Classifier not learning. Check lambda1 not drowning L_cls.")

    print(f"\n{'='*60}")
    print(f"RESULT: {checks_passed}/{checks_total} checks passed")
    if checks_passed == checks_total:
        print("✅ ALL CHECKS PASSED — Safe to run 15 epochs!")
        print("\n👉 Change EPOCHS_PHASE2 = 15 in Cell 1, then re-run from Cell 8.")
    else:
        print("❌ SOME CHECKS FAILED — Do NOT run 15 epochs.")
        print("   Diagnose above, fix code, and re-run from Cell 7.")
    print("="*60)
else:
    print("❌ train_metrics.csv not found — did Phase 2 complete without errors?")

In [ ]:
# Check evaluation metrics
if os.path.exists(metrics_csv):
    df_eval = pd.read_csv(metrics_csv, header=None, names=["Metric", "Value"])
    print("\n📊 Evaluation Metrics (metrics.csv):")
    print(df_eval.to_string(index=False))

    auc_row = df_eval[df_eval["Metric"] == "auc_roc"]
    if len(auc_row):
        test_auc = float(auc_row["Value"].values[0])
        print(f"\n🎯 Test AUC-ROC: {test_auc:.4f}")
        if test_auc >= 0.80:
            print("   → EXCELLENT")
        elif test_auc >= 0.65:
            print("   → GOOD — run more epochs")
        elif test_auc >= 0.55:
            print("   → IMPROVING — run more epochs")
        else:
            print("   → POOR — check data pipeline")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 10 — Training Curves Visualization                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
log_csv = os.path.join(OUTPUT_DIR, 'logs.csv')
if not os.path.exists(log_csv):
    print('No logs.csv found — training may not have started yet.')
else:
    df = pd.read_csv(log_csv, names=['step','tag','value'], header=0)

    df['split'] = df['tag'].str.split('/').str[0]
    df['metric'] = df['tag'].str.split('/').str[1]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    train_loss = df[(df['split']=='train') & (df['metric']=='loss')]
    if len(train_loss):
        ax.plot(train_loss['step'], train_loss['value'], label='Train Loss', color='steelblue')
        ax.set_title('Training Loss per Epoch')
        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No training loss data', ha='center', va='center', transform=ax.transAxes)

    ax = axes[0, 1]
    val_auc = df[(df['split']=='val') & (df['metric']=='auc_roc')]
    if len(val_auc):
        ax.plot(val_auc['step'], val_auc['value'], marker='o', label='Val AUC-ROC', color='darkgreen')
        ax.axhline(0.5, color='red', linestyle='--', label='Chance')
        ax.axhline(0.8, color='green', linestyle='--', alpha=0.5, label='Good (>0.8)')
        ax.set_title('Validation AUC-ROC')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('AUC-ROC')
        ax.set_ylim(0.45, 1.0)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No validation AUC data', ha='center', va='center', transform=ax.transAxes)

    ax = axes[1, 0]
    real_acc = df[(df['split']=='train') & (df['metric']=='real_acc')]
    fake_acc = df[(df['split']=='train') & (df['metric']=='fake_acc')]
    if len(real_acc) or len(fake_acc):
        if len(real_acc):
            ax.plot(real_acc['step'], real_acc['value'], marker='o', label='Real Acc', color='blue')
        if len(fake_acc):
            ax.plot(fake_acc['step'], fake_acc['value'], marker='s', label='Fake Acc', color='red')
        ax.set_title('Per-Class Training Accuracy')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_ylim(0, 1.05)
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No per-class accuracy data', ha='center', va='center', transform=ax.transAxes)

    ax = axes[1, 1]
    lr_data = df[(df['split']=='train') & (df['metric']=='lr')]
    if len(lr_data):
        ax.plot(lr_data['step'], lr_data['value'], color='purple')
        ax.set_title('Learning Rate Schedule')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('LR')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No LR data', ha='center', va='center', transform=ax.transAxes)

    plt.tight_layout()
    curves_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Training curves saved: {curves_path}')

    from IPython.display import Image, display
    display(Image(filename=curves_path))

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 11 — Evaluation Metrics Table                                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os, pandas as pd
from IPython.display import display

In [ ]:
csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
if not os.path.exists(csv_path):
    print("metrics.csv not found. Did training complete without errors?")
else:
    df = pd.read_csv(csv_path, header=None, names=["Metric", "Value"])
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").round(4)
    print("=" * 45)
    print(" EAHN Evaluation Metrics")
    print("=" * 45)
    display(df)

    auc_row = df.loc[df["Metric"] == "auc_roc", "Value"]
    if len(auc_row) and not pd.isna(auc_row.values[0]):
        v = auc_row.values[0]
        if v >= 0.80: msg = "EXCELLENT --- model is performing well."
        elif v >= 0.65: msg = "GOOD --- model is learning; more epochs will help."
        elif v >= 0.50: msg = "IMPROVING --- learning detected; run more epochs."
        else: msg = "POOR --- near chance; check data pipeline."
        print(f"\nAUC-ROC = {v:.4f} → {msg}")
    else:
        print("\nWARNING: AUC-ROC is NaN")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 12 — Detection Graphs Display                                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
graphs = [
    ("roc_curve.png", "ROC Curve"),
    ("pr_curve.png", "Precision-Recall Curve"),
    ("confusion_matrix.png", "Confusion Matrix"),
    ("confusion_matrix_norm.png", "Confusion Matrix (Normalised)"),
    ("score_distribution.png","Score Distribution"),
]

In [ ]:
available = [(f, t) for f, t in graphs
             if os.path.exists(os.path.join(OUTPUT_DIR, f))]

In [ ]:
if not available:
    print("No graph files found --- did Cell 7 produce output?")
else:
    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]
    for ax, (fname, title) in zip(axes, available):
        ax.imshow(mpimg.imread(os.path.join(OUTPUT_DIR, fname)))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    missing = [t for f, t in graphs if (f, t) not in available]
    if missing:
        print(f"Not yet generated: {missing}")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 13 — Heatmap Videos Display                                             ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os
from IPython.display import Video, HTML, display

In [ ]:
heatmap_dir = os.path.join(OUTPUT_DIR, "heatmaps")
if not os.path.isdir(heatmap_dir):
    print("No heatmaps directory found. Did training complete?")
else:
    methods = ["intrinsic", "gradcam", "rollout", "shap"]
    mp4s = sorted(f for f in os.listdir(heatmap_dir) if f.endswith(".mp4"))
    vid_ids = sorted(set(
        f.replace(f"_{m}.mp4", "")
        for f in mp4s for m in methods
        if f.endswith(f"_{m}.mp4")
    ))

    if not vid_ids:
        print("No heatmap .mp4 files found in", heatmap_dir)
    else:
        sample = vid_ids[0]
        display(HTML(f"<h3>Sample: {sample} &nbsp;|&nbsp; {len(vid_ids)} total sample(s)</h3>"))
        for method in methods:
            path = os.path.join(heatmap_dir, f"{sample}_{method}.mp4")
            if os.path.exists(path):
                display(HTML(f"<p><b>{method.upper()}</b></p>"))
                display(Video(path, embed=True, width=480))
            else:
                print(f" [{method}] not found")

╔══════════════════════════════════════════════════════════════════════════════╗
║  CELL 14 — Download Outputs                                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import os, shutil

In [ ]:
zip_out = "/kaggle/working/eahn_outputs.zip"
if os.path.exists(zip_out):
    os.remove(zip_out)

In [ ]:
shutil.make_archive("/kaggle/working/eahn_outputs", 'zip', OUTPUT_DIR)
print(f"Zipped outputs: {zip_out}")
print(f"Size: {os.path.getsize(zip_out) / (1024*1024):.1f} MB")
print("\nRight panel → Output → Download 'eahn_outputs.zip'")